# Exploratory data analysis: full-article news sentiment

This notebook describes the data behind the final study: 240 training articles and 691 test articles from Wikinews in German, English and Japanese, each labelled negative, neutral or positive.

It reads only metadata that is versioned in the repository (provenance, annotations, fold membership, the test manifest and saved test predictions). Raw article texts are kept out of Git, so nothing here needs them, and the notebook runs no model.

Labels are **AI-labelled · Codex-reviewed**. There is no independent human gold standard.

Every chart is also saved to `reports/eda/figures/`.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
NS = root / "reports/next_stage"
FIG = root / "reports/eda/figures"
FIG.mkdir(parents=True, exist_ok=True)

LABELS = ["negative", "neutral", "positive"]
LANGS = ["de", "en", "ja"]
LANG_NAMES = {"de": "German", "en": "English", "ja": "Japanese"}
COLORS = {"negative": "#c0504d", "neutral": "#9e9e9e", "positive": "#4f81bd"}
LANG_COLORS = {"de": "#2e7d32", "en": "#1565c0", "ja": "#ef6c00"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "figure.figsize": (9, 4.5),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
    "axes.titleweight": "bold", "font.size": 10,
})
pd.set_option("display.width", 140)

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    plt.show()

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

## 1. Loading the versioned records

Training rows come from `train-provenance-v1.jsonl` and `train-annotations-v1.jsonl`. The test set was drawn from 856 collected candidates (`final-test-v1/provenance.jsonl` and `annotations.jsonl`). The test manifest lists 165 of them as excluded, which leaves the 691 articles that were actually scored. The reference label is always the second-pass label.

In [ ]:
def annotation_frame(rows):
    return pd.DataFrame([{
        "id": r["id"],
        "first_label": r["first_pass"]["label"],
        "label": r["second_pass"]["label"],
        "first_ambiguous": r["first_pass"]["ambiguous"],
        "ambiguous": r["second_pass"]["ambiguous"],
        "provenance": r["provenance"],
    } for r in rows])

keep = ["id", "language", "title", "published_at", "source_keys", "event_ids", "source_license", "revision_url"]

train_prov = pd.DataFrame(jsonl(NS / "train-provenance-v1.jsonl"))
train_ann = annotation_frame(jsonl(NS / "train-annotations-v1.jsonl"))
train = train_prov[keep + ["group"]].merge(train_ann, on="id", validate="one_to_one")
assert (train_prov.set_index("id").label == train.set_index("id").label).all()

manifest = json.loads((NS / "final-test-v1/test-manifest.json").read_text())
exclusions = pd.DataFrame([{
    "id": e["id"],
    "code": c,
} for e in manifest["exclusions"] for c in e.get("codes", [e.get("code")])])

pool_prov = pd.DataFrame(jsonl(NS / "final-test-v1/provenance.jsonl"))
pool_ann = annotation_frame(jsonl(NS / "final-test-v1/annotations.jsonl"))
pool = pool_prov[keep].merge(pool_ann, on="id", validate="one_to_one")
pool["excluded"] = pool.id.isin(exclusions.id)
test = pool[~pool.excluded].copy()

train["split"], test["split"] = "train", "test"
data = pd.concat([train, test], ignore_index=True)
data["published_at"] = pd.to_datetime(data.published_at)

# The scored test set must match the frozen manifest exactly.
assert len(test) == manifest["article_rows"]
assert test.language.value_counts().to_dict() == manifest["language_counts"]
assert (test.language + ":" + test.label).value_counts().to_dict() == manifest["language_label_counts"]
print(f"train {len(train)}, test candidates {len(pool)}, excluded {pool.excluded.sum()}, scored test {len(test)}")

## 2. Dataset overview

In [ ]:
overview = (data.groupby(["split", "language"])
    .agg(articles=("id", "size"),
         first_published=("published_at", "min"),
         last_published=("published_at", "max"),
         ambiguous=("ambiguous", "sum"))
    .reindex(pd.MultiIndex.from_product([["train", "test"], LANGS])))
overview["first_published"] = overview.first_published.dt.date
overview["last_published"] = overview.last_published.dt.date
overview.loc[("all", ""), :] = [len(data), data.published_at.min().date(), data.published_at.max().date(), data.ambiguous.sum()]
overview

### Articles per language and split

The test set is almost three times the size of the training set. English is the smallest language in both splits, so English scores rest on fewer examples than German or Japanese.

In [ ]:
counts = data.groupby(["language", "split"]).size().unstack()[["train", "test"]].reindex(LANGS)
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(LANGS)); w = 0.38
for i, (split, color) in enumerate([("train", "#607d8b"), ("test", "#263238")]):
    bars = ax.bar(x + (i - 0.5) * w, counts[split], w, label=split, color=color)
    ax.bar_label(bars, padding=2)
ax.set_xticks(x, [LANG_NAMES[l] for l in LANGS])
ax.set_ylabel("articles"); ax.set_title("Articles per language and split"); ax.legend()
save(fig, "01_articles_per_language")
counts

## 3. Label distribution

Negative articles make up the largest share in every language and split. News reports on accidents, conflicts and court cases far more often than on good outcomes, and the rubric labels the outcome for the main actor.

In [ ]:
label_counts = data.groupby(["split", "language", "label"]).size().unstack(fill_value=0)[LABELS]
label_counts = label_counts.reindex(pd.MultiIndex.from_product([["train", "test"], LANGS]))
shares = label_counts.div(label_counts.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, split in zip(axes, ["train", "test"]):
    part = shares.loc[split]
    bottom = np.zeros(len(part))
    for label in LABELS:
        ax.bar([LANG_NAMES[l] for l in part.index], part[label], bottom=bottom, color=COLORS[label], label=label)
        for j, (b, v, n) in enumerate(zip(bottom, part[label], label_counts.loc[split][label])):
            ax.text(j, b + v / 2, f"{n}\n{v:.0%}", ha="center", va="center", color="white", fontsize=8)
        bottom += part[label].values
    ax.set_title(f"{split} (n={int(label_counts.loc[split].values.sum())})")
    ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[0].set_ylabel("share of articles"); axes[1].legend(loc="upper right", bbox_to_anchor=(1.32, 1))
fig.suptitle("Label distribution per language and split", fontweight="bold")
save(fig, "02_label_distribution")
label_counts

### Class imbalance and the training weights

Because negative dominates, an unweighted model could score well by mostly answering "negative". `fit_lora` in `src/next_stage_experiment.py` (line 243) weights each class by `n / (3 * count)`, so a rare class counts more in the loss. The table shows the weights the final model saw when it was trained on all 240 training articles. Inside each cross-validation fold the weights are recomputed from that fold's fit rows.

In [ ]:
y = train.label.map(LABELS.index).to_numpy()
class_counts = np.bincount(y, minlength=3)
weights = len(train) / (3 * class_counts)
weight_table = pd.DataFrame({"train_count": class_counts, "share": class_counts / len(train), "loss_weight": weights}, index=LABELS)
print(f"negative : neutral : positive in train = {' : '.join(map(str, class_counts))}")
weight_table.round(3)

## 4. Article length

Character counts are not stored in the versioned files, and raw texts are not in Git. What is stored is a Qwen token profile for every training article (`pilot/token-profile-development-v2-qwen.json`, article text only, without the chat prompt). No length record exists for the test articles. The dashed line marks the 4096-token article limit in `qwen_tokens()`: a longer article is rejected instead of being cut.

In [ ]:
profile = json.loads((NS / "pilot/token-profile-development-v2-qwen.json").read_text())
minilm = json.loads((NS / "pilot/token-profile-development-v2-minilm.json").read_text())
tok = pd.DataFrame(profile["items"])[["id", "tokens"]].rename(columns={"tokens": "qwen_tokens"})
tok = tok.merge(pd.DataFrame(minilm["items"])[["id", "tokens", "chunks"]].rename(columns={"tokens": "minilm_tokens", "chunks": "minilm_chunks_128"}), on="id")
train_len = train.merge(tok, on="id", validate="one_to_one")
assert len(train_len) == len(train)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
bins = np.linspace(0, train_len.qwen_tokens.max() * 1.05, 30)
for lang in LANGS:
    axes[0].hist(train_len.loc[train_len.language == lang, "qwen_tokens"], bins=bins, alpha=0.55, color=LANG_COLORS[lang], label=LANG_NAMES[lang])
axes[0].set_xlabel("Qwen tokens per article"); axes[0].set_ylabel("articles"); axes[0].legend()
axes[0].set_title("Length histogram (train)")
axes[1].boxplot([train_len.loc[train_len.language == l, "qwen_tokens"] for l in LANGS], tick_labels=[LANG_NAMES[l] for l in LANGS])
axes[1].axhline(4096, ls="--", color="black", lw=1); axes[1].text(0.55, 4096, " 4096-token limit", va="bottom", fontsize=8)
axes[1].set_ylabel("Qwen tokens"); axes[1].set_title("Length per language (train)")
save(fig, "03_article_length_tokens")
train_len.groupby("language")[["qwen_tokens", "minilm_tokens", "minilm_chunks_128"]].describe().T.round(0)

Even the longest training article stays well below the limit, so no training article was rejected for length. The MiniLM baselines read articles in 128-token chunks, which is why their chunk counts are listed as well.

In [ ]:
print("longest training article:", int(train_len.qwen_tokens.max()), "Qwen tokens")
print("articles over 4096 tokens:", int((train_len.qwen_tokens > 4096).sum()))
print("share of label by length quartile:")
train_len["length_quartile"] = pd.qcut(train_len.qwen_tokens, 4, labels=["Q1 short", "Q2", "Q3", "Q4 long"])
pd.crosstab(train_len.length_quartile, train_len.label, normalize="index").round(2)[LABELS]

## 5. Publication dates

Articles span about two decades of Wikinews. Training and test cover a similar range, so the test set is not simply a later time slice of the same data.

In [ ]:
years = data.assign(year=data.published_at.dt.year).groupby(["year", "split"]).size().unstack(fill_value=0)[["train", "test"]]
fig, ax = plt.subplots(figsize=(10, 4))
years.plot(kind="bar", stacked=True, color=["#607d8b", "#263238"], ax=ax, width=0.85)
ax.set_xlabel("publication year"); ax.set_ylabel("articles"); ax.set_title("Articles by publication year")
save(fig, "04_publication_years")

lang_years = data.assign(year=data.published_at.dt.year).groupby(["year", "language"]).size().unstack(fill_value=0)[LANGS]
fig, ax = plt.subplots(figsize=(10, 3.8))
for lang in LANGS:
    ax.plot(lang_years.index, lang_years[lang], marker="o", ms=3, color=LANG_COLORS[lang], label=LANG_NAMES[lang])
ax.set_xlabel("publication year"); ax.set_ylabel("articles"); ax.set_title("Articles per year and language (train + test)"); ax.legend()
save(fig, "05_publication_years_language")

## 6. Sources

All articles are Wikinews pages in the respective language edition under CC BY 2.5 or CC BY 4.0. Each page cites outside reports, recorded as `source_keys`. The chart counts how often each cited domain appears, which shows which outlets the stories were built on. Many pages also store social-media share buttons and links back to Wikinews as source keys. Those are not outlets, so the code below removes them first.

In [ ]:
def domain(key):
    host = urlparse("http:" + key if key.startswith("//") else key).netloc.lower()
    return host[4:] if host.startswith("www.") else host

cited = data[["split", "language", "source_keys"]].explode("source_keys").dropna()
# Some pages also record "share this article" links (reddit submit, bsky compose, mastodon share).
# They and internal Wikinews links are dropped before counting outlets.
share = cited.source_keys.str.contains(r"/submit\?|/intent/|/share\?|wikinews", regex=True)
print(f"share links dropped: {int(share.sum())} of {len(cited)} recorded source keys")
cited = cited[~share].copy()
cited["domain"] = cited.source_keys.map(domain)
print("cited sources per article, share links excluded:")
print(cited.groupby([cited.index]).size().reindex(data.index, fill_value=0).groupby(data.language).describe()[["mean", "50%", "max"]].round(1))
print("\nlicences:", data.source_license.value_counts().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, lang in zip(axes, LANGS):
    top = cited[cited.language == lang].domain.value_counts().head(10)[::-1]
    ax.barh(top.index, top.values, color=LANG_COLORS[lang])
    ax.set_title(f"{LANG_NAMES[lang]}: top cited domains"); ax.set_xlabel("citations")
save(fig, "06_cited_source_domains")

## 7. Event groups and cross-validation folds

Several Wikinews articles can report on the same event, for example a follow-up on an earthquake. If one of them sits in the fit part and another in the dev part, a model can score well by recognising the story rather than reading the sentiment. The project therefore assigns articles to groups and keeps every group whole within one fold. `pilot/cv-membership-v1.json` holds the three folds used for model selection.

In [ ]:
group_sizes = train.groupby("group").size()
print(f"training groups: {group_sizes.size}; groups with more than one article: {(group_sizes > 1).sum()}; largest group: {group_sizes.max()} articles")
print("articles linked to more than one event id:", int((train.event_ids.map(len) > 1).sum()))

folds = json.loads((NS / "pilot/cv-membership-v1.json").read_text())
gid = train.set_index("id").group
rows = []
for f in folds:
    fit, dev = set(f["fit_ids"]), set(f["dev_ids"])
    assert not fit & dev
    assert not set(gid[list(fit)]) & set(gid[list(dev)]), "a group crosses fit and dev"
    for part, ids in [("fit", fit), ("dev", dev)]:
        sub = train[train.id.isin(ids)]
        for (lang, label), n in sub.groupby(["language", "label"]).size().items():
            rows.append({"fold": f["fold"], "part": part, "language": lang, "label": label, "n": n})
fold_df = pd.DataFrame(rows)
print("checked: no event group crosses fit and dev in any fold")

dev = fold_df[fold_df.part == "dev"].pivot_table(index=["fold", "language"], columns="label", values="n", aggfunc="sum", fill_value=0)[LABELS]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, fold in zip(axes, sorted(fold_df.fold.unique())):
    part = dev.loc[fold].reindex(LANGS)
    bottom = np.zeros(len(part))
    for label in LABELS:
        ax.bar([LANG_NAMES[l] for l in LANGS], part[label], bottom=bottom, color=COLORS[label], label=label)
        bottom += part[label].values
    ax.set_title(f"fold {fold} dev (n={int(part.values.sum())})")
axes[0].set_ylabel("articles"); axes[-1].legend(loc="upper right", bbox_to_anchor=(1.45, 1))
fig.suptitle("Dev composition of the three grouped folds", fontweight="bold")
save(fig, "07_fold_composition")
fold_df.pivot_table(index="fold", columns="part", values="n", aggfunc="sum")

The three dev parts have almost the same size and the same language and label mix, so the out-of-fold scores of different folds can be averaged fairly. The English dev parts hold only three or four neutral articles each, which makes English neutral recall in cross-validation very noisy.

For the test set, a review compared every candidate against all earlier project articles by character n-gram similarity and joined related stories into groups before any model saw the test (`final-test-v1/group-review.json`).

In [ ]:
review = json.loads((NS / "final-test-v1/group-review.json").read_text())
pairs = pd.DataFrame(review["pairs"])
print(f"reviewed similar pairs: {len(pairs)}; protected ids compared against: {review['protected_unique_ids']}")
print(pairs.decision.value_counts().to_string())
print(f"observed groups among the {manifest['article_rows']} test articles: {manifest['observed_groups']}")

fig, ax = plt.subplots(figsize=(8, 3.8))
for decision, color in [("separate_observed_event", "#90a4ae"), ("join_conservative_component", "#d84315")]:
    ax.hist(pairs.loc[pairs.decision == decision, "similarity"], bins=30, alpha=0.7, color=color, label=decision.replace("_", " "))
ax.set_xlabel("char n-gram TF-IDF similarity"); ax.set_ylabel("pairs"); ax.legend()
ax.set_title("Test group review: similarity of reviewed pairs")
save(fig, "08_group_review_similarity")

High similarity does not decide the outcome on its own: some close pairs are separate events (two different storms in the same region) and some lower-scoring pairs were joined after reading. This is why the grouping was reviewed by reading the pairs instead of using a similarity threshold.

## 8. Annotation quality

Each article was labelled twice by the same AI annotator (Codex), and the second pass is the reference. Agreement between the passes says how stable the label is, but both passes come from one annotator, so it is not independent validation.

In [ ]:
def agreement(df):
    return pd.Series({
        "articles": len(df),
        "same_label": int((df.first_label == df.label).sum()),
        "agreement": (df.first_label == df.label).mean(),
        "changed": int((df.first_label != df.label).sum()),
        "ambiguous_second_pass": int(df.ambiguous.sum()),
        "ambiguous_share": df.ambiguous.mean(),
    })

quality = pd.concat({
    "train": train.groupby("language").apply(agreement),
    "test (scored)": test.groupby("language").apply(agreement),
    "test candidates (all 856)": pool.groupby("language").apply(agreement),
})
quality.loc[("train", "all"), :] = agreement(train)
quality.loc[("test (scored)", "all"), :] = agreement(test)
quality.loc[("test candidates (all 856)", "all"), :] = agreement(pool)
quality = quality.sort_index()
print("manifest check, all candidates:", manifest["same_annotator_label_agreement"])
quality.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
cm = pd.crosstab(test.first_label, test.label).reindex(index=LABELS, columns=LABELS, fill_value=0)
im = axes[0].imshow(cm.values, cmap="Blues")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cm.values[i, j], ha="center", va="center", color="white" if cm.values[i, j] > cm.values.max() / 2 else "black")
axes[0].set_xticks(range(3), LABELS); axes[0].set_yticks(range(3), LABELS); axes[0].grid(False)
axes[0].set_xlabel("second pass (reference)"); axes[0].set_ylabel("first pass")
axes[0].set_title("First vs second pass, scored test set")

amb = data.groupby(["split", "label"]).ambiguous.mean().unstack()[LABELS].reindex(["train", "test"])
amb.T.plot(kind="bar", ax=axes[1], color=["#607d8b", "#263238"], rot=0)
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[1].set_ylabel("flagged ambiguous"); axes[1].set_title("Ambiguity flag by reference label")
save(fig, "09_annotation_quality")

Train labels were almost never changed (one article out of 240), while the test passes disagree on 31 of the 691 scored articles. 24 of those 31 changes move an article into or out of neutral, and the most common single change is positive to neutral (13 articles). Neutral is the class with the fuzziest boundary, which matches where the model struggles in section 10. The ambiguity flag was used much more often in training (14.6 %) than in the scored test set (4.1 %), and in the test set almost only for Japanese.

In [ ]:
changes = pd.concat([train, test]).query("first_label != label")
changes.groupby(["split", "first_label", "label"]).size().rename("articles").to_frame()

## 9. Excluded test candidates

Of the 856 collected test candidates, 165 were excluded before any test inference. Most were related to an article the project had already used (`related_protected_component`), so keeping them would leak a known story into the test. Others had a rights review that was not finished, a quality problem in the extracted text, or an unresolved label. One article can carry more than one code.

In [ ]:
ex = exclusions.merge(pool[["id", "language"]], on="id")
reason_table = ex.groupby(["code", "language"]).size().unstack(fill_value=0).reindex(columns=LANGS, fill_value=0)
reason_table["total"] = reason_table.sum(axis=1)
reason_table = reason_table.sort_values("total")

fig, ax = plt.subplots(figsize=(9, 3.8))
left = np.zeros(len(reason_table))
for lang in LANGS:
    ax.barh(reason_table.index, reason_table[lang], left=left, color=LANG_COLORS[lang], label=LANG_NAMES[lang])
    left += reason_table[lang].values
ax.set_xlabel("excluded candidates"); ax.set_title("Why test candidates were excluded"); ax.legend()
save(fig, "10_test_exclusions")
print("excluded articles:", ex.id.nunique(), "| per language:", ex.drop_duplicates("id").language.value_counts().reindex(LANGS).to_dict())
reason_table.sort_values("total", ascending=False)

The table compares the label mix of excluded and scored candidates. Exclusion removed somewhat more neutral and fewer positive articles than it kept, so the scored test set is slightly more positive than the full candidate pool. Selection followed source order and grouping rules, never the label.

In [ ]:
pd.crosstab(pool.excluded.map({True: "excluded", False: "scored"}), pool.label, normalize="index").round(3)[LABELS]

## 10. Where the selected model makes mistakes

`final-test-v1.json` holds the single test pass of all four candidates, and `error-analysis-v1.json` lists every test article on which at least one model disagreed with the reference. Here we look at the selected model, Qwen3-4B with LoRA (`qwen-lora`).

In [ ]:
final = {m["model"]: m for m in json.loads((NS / "final-test-v1.json").read_text())["models"]}
q = final["qwen-lora"]
print(f"qwen-lora primary score (mean of DE and EN macro-F1) {q['primary_score']:.4f}")
print(f"qwen-lora pooled macro-F1 over all 691 {q['macro_f1']:.4f}, accuracy {q['accuracy']:.4f}")
print("primary score of all candidates:", {k: round(m["primary_score"], 4) for k, m in final.items()})
print("macro-F1 per language:", {l: round(q[f"macro_f1_{l}"], 4) for l in LANGS})

cases = pd.DataFrame(json.loads((NS / "error-analysis-v1.json").read_text())["cases"])
cases["language"] = cases.id.str.split(":").str[1]
cases["qwen_lora"] = cases.predictions.map(lambda p: p["qwen-lora"])
wrong = cases[cases.qwen_lora != cases.reference]
assert set(cases.id) <= set(test.id)

err_rate = (wrong.groupby(["language", "reference"]).size()
            / test.groupby(["language", "label"]).size().rename_axis(["language", "reference"])).unstack().reindex(LANGS)[LABELS].fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
cmq = np.array(q["confusion_matrix"])
axes[0].imshow(cmq, cmap="Blues")
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, cmq[i, j], ha="center", va="center", color="white" if cmq[i, j] > cmq.max() / 2 else "black")
axes[0].set_xticks(range(3), LABELS); axes[0].set_yticks(range(3), LABELS); axes[0].grid(False)
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("reference"); axes[0].set_title("qwen-lora confusion matrix, 691 test articles")
err_rate.plot(kind="bar", ax=axes[1], color=[COLORS[l] for l in LABELS], rot=0)
axes[1].set_xticklabels([LANG_NAMES[l] for l in LANGS])
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
axes[1].set_ylabel("error rate"); axes[1].set_title("qwen-lora error rate by reference label")
save(fig, "11_qwen_lora_errors")
print("qwen-lora errors:", len(wrong), "of", len(test))
err_rate.round(3)

Negative articles are recognised most reliably. Of the 101 errors, 30 are on negative articles, 36 on neutral and 35 on positive, although neutral and positive together make up only about 40 % of the test set. Most neutral errors are predicted positive (24 of 36) and most positive errors are predicted neutral (25 of 35). Neutral is also where most annotation changes happened, so part of the error reflects uncertainty in the reference itself. The German positive class has the highest error rate. Articles flagged as ambiguous are much harder, as the table below shows.

In [ ]:
amb_ids = set(test.loc[test.ambiguous, "id"])
pd.Series({
    "ambiguous test articles": len(amb_ids),
    "qwen-lora errors on them": int(wrong.id.isin(amb_ids).sum()),
    "error rate, ambiguous": wrong.id.isin(amb_ids).sum() / max(len(amb_ids), 1),
    "error rate, not ambiguous": (~wrong.id.isin(amb_ids)).sum() / (len(test) - len(amb_ids)),
}).round(3)

## 11. Limitations

- The labels come from one AI annotator (Codex) in two passes. The team checked them after the measurements, but there is no independent human gold standard, so all scores measure agreement with this reference.
- English has 71 training and 118 test articles, with only 10 neutral training and 14 neutral test articles. English results per class should be read with wide error bars.
- Article length is known only for the training set and only in tokens. The versioned files have no character counts and no length record for the test set.
- The event grouping relies on n-gram retrieval and reading. It cannot rule out every semantic near-duplicate or overlap with the language model's pretraining data.
- All articles come from Wikinews. Commercial news sites write differently, so performance on arbitrary URLs in the demo may be lower than on this test set.